### **Student Information**
GitHub ID: brendatan99

Kaggle name: brendatt

Kaggle private scoreboard snapshot: 

![pic_ranking.png](./pics/pic_ranking.png)

*Your name and student ID are deliberately not asked for here: this notebook goes into your repository, which is public. They belong in the git-ignored `config/.env`, where the other two notebooks read them from, and your NTU COOL submission is what ties the repository link back to you.*

---

# **Instructions**

This lab has **three parts**: exercises in the Master notebook, exercises and an agent conversation in the AgentPipeline notebook, and an internal Kaggle competition documented in this Homework notebook. Together they add up to **100 points**.

**Environment recommendations to solve lab 2:**
- **Part 1 and 2 exercises:** Everything can be run with only CPU.
- **Part 3 exercises:** For the competition you will probably need GPU to train your models, so it is recommended to use Colab or Kaggle if you don't have a laptop with a dedicated GPU.

## **Part 1 (40 pts):**

1. __Main Exercises (35 pts):__ Do the **take home exercises** from Sections: `1. Data Preparation` to `8. Information Retrieval and RAG`, in the DM2026-Lab2-Master Notebook. Total: `13 exercises`. Commit your code and submit the repository link to NTU Cool.

2. **Code Comments (5 pts):** **Tidy up the code in your notebook**. 

## **Part 2 (30 pts):**

1. **Agent Prompt (20 pts):** Interact with the Agent using appropriate prompts to guide it through and complete all the required tasks.

2. **Main Exercises (10 pts):** Do the remaining **take home exercises** from DM2026-Lab2-AgentPipeline Notebook. Total: `3 exercises required`. Commit your code and submit the repository link to NTU Cool.

## **Part 3 (30 pts):**

1. **Kaggle Competition Participation (15 pts):** Submit your predictions to the [Data Mining Class Kaggle Competition](https://www.kaggle.com/t/d99def2e62194b0d8f633267c32e6d1e) by **November 22**. Check Kaggle for the exact closing time. The competition concerns emotion recognition on Twitter; participation points use the Private Leaderboard ranking:
    - If your submission meets or exceeds the **baseline score (F1 score: 0.5)**, and there are no    significant issues with your report or code, you will automatically receive a base score of 5 points of the 15 points.

        The remaining 10 points will be determined by your ranking among participants who meet or exceed the baseline:

        |Ranking  |Additional Points  |
        | --- | --- |
        |Top 10% |+10 |
        |Top 20% |+9 |
        |Top 30% |+8 |
        |Top 40% |+7 |
        |Top 50% |+6 |
        |Top 60% |+5 |
        |Top 70% |+4 |
        |Top 80% |+3 |
        |Top 90% |+2 |
        |<Top 90% |+1 |

    - If your submission does not reach the baseline score, your grade will be lower than 5. The score will range from 0 to 4 points, depending on your Private Leaderboard ranking and the relative performance of students whose submissions are also below the baseline.

2. **Competition Report (15 pts)** A report section to be filled in inside this notebook in Markdown Format, we already provided you with the template below. You need to describe your work developing the model for the competition. The report should include a section describing briefly the following elements: 
* All of your AI prompts and what you decide do use.
* Your preprocessing steps.
* The feature engineering steps.
* Explanation of your model.

[Markdown Guide - Basic Syntax](https://www.markdownguide.org/basic-syntax/)

**`Things to note for Part 3:`**

* **The code used for the competition should be in this Jupyter Notebook File** `DM2026-Lab2-Homework.ipynb`.

* **Push the code used for the competition to your repository**.

* **The code should have a clear separation for the same sections of the report, AI prompts, preprocessing, feature engineering and model explanation. Briefly comment your code for easier understanding, we provide a template at the end of this notebook.**

* Save a screenshot of your Kaggle Private Leaderboard ranking as `pics/pic_ranking.png` and keep the image link in the Student Information cell at the top of this notebook.

The Kaggle competition closes on **November 22**. The three days afterward are for completing this notebook's Markdown report, checking your files, and submitting the lab by **November 25 at 11:59 pm (Wednesday)**.

Push your work to your repository, then submit its link and a ZIP on the corresponding NTU COOL assignment. The ZIP must contain `DM2026-Lab2-Master.ipynb`, `DM2026-Lab2-AgentPipeline.ipynb`, `DM2026-Lab2-Homework.ipynb`, both encrypted logs in `session_logs/`, and `pics/pic_ranking.png`. Name it with your student ID and your name. The two logs are git-ignored, so they are not in your repository and the ZIP is the only copy that reaches us: open it before uploading and check both are inside. If you miss them, you can upload them on NTU COOL until **November 26 at 11:59 pm**. See `README.md` for the exact file paths and setup steps.

---

Next you will find the template report with some simple markdown syntax explanations, use it to structure your content.

You can delete the syntax suggestions after you use them.

---

***

# **Project Report**

---

## 1. Model Development

### 1.1 Preprocessing Steps

The competition data comes in three files. `final_posts.json` has the 4,800 posts, `data_identification.csv` splits them into 3,360 training and 1,440 test rows, and `emotion.csv` has the training labels. I merged the three files on `post_id` and checked the result with assertions instead of assuming the merge worked: all 4,800 rows match across the files, `post_id` is unique, and no text is missing. If a row had been dropped silently here, every later number would be wrong.

Two details from this check shaped the rest of the pipeline. First, the label column in `emotion.csv` is called `label`, not `emotion`, and it has exactly six values: `anger`, `disgust`, `fear`, `joy`, `sadness` and `surprise`. The submission code checks every prediction against this exact list, so a spelling mistake cannot reach Kaggle. Second, the test rows in the merged table are not in the same order as `sampleSubmission.csv`. If I wrote the predictions in the table's own order, almost every label would land on the wrong row, and the file would still look fine. So every prediction set is joined to `sampleSubmission.csv` by ID before it is written.

The training data is imbalanced and the text lengths are skewed. Joy is the largest class at 26.04% and fear the smallest at 8.57%, a ratio of 3.04. The competition metric is macro F1, which gives every class the same weight, so accuracy is not a good guide. Text length is also very uneven: the median post has 13 words, but the longest has 508. To stop long posts from dominating the features just by repeating words, I used sublinear term frequency (`sublinear_tf=True`) in every TF-IDF view.

The most important preprocessing step was building a validation split without leakage. I normalised the text (lowercase, collapsed spaces) to find duplicates. In the training set, 50 texts appear more than once, covering 100 rows, and 16 of these groups have conflicting labels. A normal `StratifiedKFold` would put copies of the same post in both the training and validation folds, and the validation score would be inflated by memorisation. So I split the data with `StratifiedGroupKFold`, grouping by the MD5 hash of the normalised text. No group spans more than one fold, and the class shares stay within 0.3 percentage points across the five folds. I saved this split to `folds.csv` and used it for every experiment, so all scores, from the linear models to the transformers, can be compared directly.

Two more facts about the data were recorded. First, the 16 duplicate groups with conflicting labels put a ceiling on any model's score. I kept them in the training set, because removing them would raise the validation score without making the model better on new text. Second, 59 texts appear in both the training and the test set, always with a single label in the training set, so the answers for 4.1% of the test rows are already known. At first I did not use this, because my fine-tuned model already got 55 of the 59 right. In my final entry without the emotion checkpoints I did use it (Section 3.8): those rows take their training label, and this changes 3 rows of that blend.

### 1.2 Feature Engineering Steps

I built three feature sets and tested each one on the fixed folds. Each one looks at the tweets at a different level.

**Word-level features.** A word TF-IDF over unigrams and bigrams (`min_df=2`, `sublinear_tf=True`) picks up emotion words directly, so the classifier can give weight to words like *furious*, *terrified* and *blessed*. On its own it reached an out-of-fold macro F1 of 0.4730.

**Character-level features.** Tweets often contain informal spelling and stretched words. A word tokenizer treats *angryyy* and *angry* as two unrelated tokens. In the Master notebook, FastText sub-word vectors gave *angryyy* and *angry* a cosine similarity of 0.984, against 0.624 for the swapped-letter typo *anrgy*, so sub-word pieces handle repeated letters well even if they do not fix every typo. A `char_wb` TF-IDF over 2- to 5-character n-grams reached 0.5045 on its own, better than the word view, and the two together reached 0.5146.

**Sentence embeddings.** Sparse n-grams cannot tell that two different words mean the same thing, such as *afraid* and *scared*. In earlier experiments, Word2Vec trained on our own data gave this pair a similarity of only 0.482, while the pre-trained MiniLM model gave 0.845. Adding the 384-dimensional MiniLM sentence embeddings (L2-normalised) to the two TF-IDF matrices raised the out-of-fold macro F1 from 0.5146 to 0.5708. This gain of 0.0562 is about 81 test rows, much more than any classifier setting changed. The combined feature matrix has 35,420 columns.

**Fine-tuned transformer.** With TF-IDF and fixed embeddings, the features are decided before training. Fine-tuning a transformer learns the text representation and the classifier together. 3,360 rows are far too few to learn a language model from scratch, so I started from `cardiffnlp/twitter-roberta-base-2022-154m`, a model pre-trained on 154 million tweets. The text is normalised the same way as in its pre-training: user handles become `@user` and links become `http`.

**Emotion-trained checkpoints (used in one of my two final entries).** The tweet model knows how people write on Twitter, but it was never taught what an emotion is. Some public checkpoints were. `j-hartmann/emotion-english-distilroberta-base` and its RoBERTa-large version were trained on six English emotion datasets with seven classes, our six plus *neutral*. Loading them the usual way throws away their classification head. I kept the head and took the six rows that match our labels, looked up by name in `config.id2label`, so fine-tuning starts from a classifier that can already tell *anger* from *disgust* to some degree. Three more emotion checkpoints were loaded the same way (Section 3.6). These models scored well, but they had been trained on data that contains about half of the test texts with their labels, so I use them in only one of my two final entries. Section 1.3 explains this.

The sparse features and the transformers treat emoji differently. The default token pattern of `TfidfVectorizer`, `(?u)\b\w\w+\b`, keeps only word characters, so punctuation and emoji are dropped. A post that shows anger only with emoji gives the linear model nothing to work with. A byte-level BPE tokenizer keeps emoji as tokens, so the transformer can see them. I tested this instead of assuming it: turning on BERTweet's emoji-to-text normaliser changed the out-of-fold macro F1 by −0.0018, which suggests the transformer was already using the raw emoji.

### 1.3 Explanation of Your Model

**Final models.** I have two final entries, and this notebook produces both of them. The first is the seven-model blend of Section 3.7, which includes the emotion-trained checkpoints of Section 3.6. The second is a blend of three models that learned only from the competition data and from text without emotion labels (Section 3.5), plus one simple rule that uses the training labels (Section 3.8). Both blends use temperature calibration and were evaluated on the same five frozen folds. Why I keep one entry of each kind is explained below. All numbers come from the notebook run.

| Final entry without the emotion checkpoints | Out-of-Fold Macro F1 |
| --- | ---: |
| `twitter-roberta-base-2022-154m` (average of 3 seeds) | 0.6120 |
| `twitter-roberta-large-2022-154m` | 0.6101 |
| TF-IDF (word + char) + MiniLM with Linear SVM | 0.5708 |
| **Three-model blend, Section 3.5 (nested CV)** | **0.6236** |

The blend is 0.0116 better than its best single model, which is about 17 test rows. The weights are 0.25 for each transformer and 0.50 for the linear model. Then Section 3.8 handles the 59 test tweets that also appear word for word in the training set with only one label. For these tweets I use the training label. The blend already agrees with 56 of them, so the rule changes only 3 rows.

| Emotion checkpoints (Section 3.6) and their blends (Section 3.7) | Out-of-Fold Macro F1 |
| --- | ---: |
| `j-hartmann/emotion-english-distilroberta-base` (average of 3 seeds) | 0.6262 |
| `j-hartmann/emotion-english-roberta-large` (average of 3 seeds) | 0.6242 |
| `SamLowe/roberta-base-go_emotions` | 0.6222 |
| `cardiffnlp/twitter-roberta-large-emotion-latest` | 0.6059 |
| `michellejieli/emotion_text_classifier` | 0.6014 |
| Four-model blend with the two j-hartmann models (nested) | 0.6438 |
| **Seven-model blend, Section 3.7 (nested), final entry** | **0.6536** |

**Emotion checkpoints and the TAs' reply.** These checkpoints raised the nested score from 0.6236 to 0.6536. A blend that also included my own GoEmotions-trained models (see 1.4) reached 0.6545 nested and 0.6539 on the public leaderboard. On 17 November I compared the competition texts with public emotion datasets. I found that 48.8% of the test texts, and about half of the training texts, come from GoEmotions. On the training set, the most common GoEmotions label matches the competition label 95.3% of the time. According to their model cards, the j-hartmann models were trained on six datasets that include GoEmotions, SamLowe's model was trained on GoEmotions only, and michellejieli's model starts from j-hartmann's. This means these models had already seen about half of my test texts, together with labels that are almost the same as the answers.

The scores show the same thing (measured with the script versions of the same models). On the 1,700 training rows that are not from GoEmotions, every checkpoint scored lower than the twitter-roberta base model (0.523 to 0.530, against 0.547). All of their advantage came from the 1,660 rows that are from GoEmotions (0.674 to 0.705, against 0.667). I also trained my own models on GoEmotions after removing every competition text from it. They showed the same pattern, and the closest remaining text was still 0.698 similar to a competition text, so I could not be sure the removal was complete. So part of the gain from these models probably comes from the overlap with GoEmotions.

Because of this, I asked the TAs whether I could use these models. In my email I described the overlap and also my own GoEmotions-trained models. The TAs replied that it is fine as long as I train my own model with the data provided by the competition. Every model in the Section 3.7 blend was fine-tuned by me on the competition data, so one of my two final entries uses that blend. The other final entry uses only models that learned from the competition data and from unlabelled text (Section 3.8). I report both. The gap between them, 0.6536 against 0.6236 nested (about 43 of the 1,440 test rows) and 0.6576 against 0.6244 on the public leaderboard, shows how much the emotion checkpoints added, and part of this gain probably comes from the overlap with GoEmotions. I did not use the 0.6545 blend as a final entry, because its main models were trained with scripts outside this notebook; the Section 3.7 blend is only 0.0009 lower out of fold, about one test row, and can be reproduced here.

**Multi-seed averaging.** A fine-tuned transformer gives slightly different results each time, because the classification head starts from random values and the data is shuffled. With three seeds, the base model scored 0.6015, 0.5995 and 0.6102, a spread of 0.0107. Averaging the outputs of the three models raised the score to 0.6120.

**Temperature calibration.** The linear SVM outputs decision margins, while the transformers output logits, so their values are on different scales. In my first blend I skipped calibration, and the search gave a weight of 0.85 to the linear SVM, the weakest model. This happened because its values were small, so after softmax its probabilities were almost flat, and the search needed a large weight to make it count. To fix this, I fitted one temperature for each model by minimizing out-of-fold log loss (1.50, 1.87 and 0.28). After calibration, the weights show how much each model really helps.

**Nested cross-validation.** If I choose the blend weights and score them on the same out-of-fold predictions, the score is too optimistic (0.6288). So I chose the weights on four folds, scored them on the fifth, and repeated this for all five folds. This gives 0.6236, which is 0.0052 lower. The rule in Section 3.8 cannot be tested this way, because the folds keep identical texts in the same fold, so no training fold ever contains the answer for a validation row. I only expect it to affect the 3 rows it changes.

**Model size.** The large model (355M parameters) scored 0.6101, slightly lower than the base model (125M parameters, 0.6120), even though it took about four times longer to train. I saw the same thing in the Master notebook, where a neural network with about 545,000 parameters did not beat a linear SVM with about 17,000 parameters on the same features. With only 3,360 training rows, the amount of data seems to be the limit, not the size of the model.

**Error analysis.** The confusion matrix of the linear model shows two main problems. First, 24.2% of disgust tweets are predicted as anger, but only 9.2% of anger tweets are predicted as disgust. Anger has 48% more training examples than disgust, so when the two are hard to tell apart, the model leans toward anger. Second, the model predicts joy too often: 952 joy predictions against 875 real joy tweets, with 316 tweets from other classes predicted as joy. Macro F1 gives every class the same weight, so mistakes on the smaller classes lower the score a lot.

All four single-model runs also had their lowest score on Fold 4 (0.5779, 0.5715, 0.5915 and 0.5796, compared with 0.60 to 0.63 on the other folds). Since different models did worst on the same fold, I think this fold simply contains harder examples, and the training itself is not unstable.

**Out-of-fold scores and the public leaderboard.**

| Submission | Out-of-Fold | Public LB | Difference |
| --- | ---: | ---: | ---: |
| TF-IDF + char + MiniLM, Linear SVM | 0.5708 | 0.5672 | −0.0036 |
| Fine-tuned RoBERTa-base (single seed) | 0.6053 | 0.6233 | +0.0180 |
| Calibrated blend (first version) | 0.6204 | 0.6364 | +0.0160 |
| Calibrated blend (seed-averaged base) | 0.6216 | 0.6330 | +0.0114 |
| Calibrated blend, three models (Section 3.5) | 0.6236 | 0.6223 | −0.0013 |
| Section 3.5 plus the duplicate rule (Section 3.8, final) | 0.6236 | 0.6244 | +0.0008 |
| Seven-model blend with the emotion checkpoints (Section 3.7, final) | 0.6536 | 0.6576 | +0.0040 |

My local scores and public scores do not always agree. Locally, the three calibrated blends differ by only 0.0032 on 3,360 rows (about 11 rows). On the public leaderboard they differ by 0.0141, and their order is reversed. However, the public leaderboard uses only about 432 test rows, so 0.0141 is only about 6 predictions. I think these differences are mostly random noise, so I ranked my models by the out-of-fold score and used the public score only to check the file format. For the final two entries, I only considered submissions that this notebook produces: the Section 3.7 seven-model file, which uses the emotion checkpoints, and the Section 3.8 file, which does not. I did not use the public score to choose between submissions. On the private leaderboard, which uses the other 70% of the test set, the Section 3.7 entry scored 0.6577 and the Section 3.8 entry scored 0.6444. The gap there (0.0133) is smaller than the nested gap (0.0300), but the order is the same.

**Limitations.** There are three main limitations. First, 16 groups of identical texts have different labels, so no model can get all of them right. Second, the gap of about 0.03 between my two final entries mixes two things: the overlap with GoEmotions and real knowledge about emotions. I believe a large part comes from the overlap, but I could not separate the two. Third, the blend weights were checked with nested cross-validation, but my other choices (which models and which settings) were also based on the same 3,360 rows. So the nested scores of 0.6536 and 0.6236 are estimates, with about ±0.01 of variation between folds.

### 1.4 AI Prompts, what and why you decide to use that technique that suggest by AI

I used Claude Code as a pair programmer for this competition. For each topic below, I give the prompt I wrote (I wrote my prompts in Chinese; these are my translations), what Claude suggested, the result on my own folds, and my decision. I also include the suggestions I rejected, because knowing what does not work was often as useful as the suggestions that did.

**Representation.** *Prompts (11 and 13 November):* "Tonight, take the Kaggle task from the data checks to a first submission." and "While I finish the AgentPipeline notebook, keep training models and send new predictions." Claude first suggested a word TF-IDF over unigrams and bigrams. I used it: on its own it reached 0.4730 out of fold, and `sublinear_tf` stopped posts of up to 508 words from dominating just by repeating words. Then I added character n-grams and MiniLM sentence embeddings, and kept both because each one helped on the same folds. Character n-grams alone reached 0.5045, better than the word view, and the two together reached 0.5146. Adding MiniLM raised this to 0.5708, a gain of 0.0562 or about 81 test rows, much more than any classifier setting changed. The last suggestion here was to stop building features by hand and fine-tune a transformer pre-trained on 154 million tweets. I used it, because it raised the out-of-fold score from 0.5708 to 0.6120, about 59 test rows. This was the largest single improvement in the project.

**Class imbalance.** *Prompts (11 and 14 November):* the first-submission prompt above, and later "Next, run the decision-offset experiment for macro F1." Claude suggested `class_weight="balanced"` for the linear model. On my folds it changed the out-of-fold score from 0.5096 to 0.5129, a difference of only 0.0033. That is less than a fifth of the between-fold standard deviation (0.0167), so random variation alone could produce it. I kept it, because it does no harm and macro F1 rewards the small classes, but I do not count it as a real improvement. Later Claude suggested changing the decision rule itself, because `argmax` maximises accuracy, not macro F1. I tested two versions in separate script runs and rejected both. Six free per-class multipliers raised the nested score by only 0.0022, about 3 rows, winning on two folds and losing on two. The fitted multipliers also changed a lot between folds (the one for joy went from 0.52 to 1.05), so the multipliers were fitting noise. A one-parameter version, which divides the probabilities by the class prior raised to a power, chose a power of 0.02 on every fold. When it could not overfit, it chose to change almost nothing. I took this as a sign that temperature calibration had already done this job. I never tested resampling, so I have no evidence about it either way.

**Model capacity and stability.** *Prompt (13 November):* "Keep fine-tuning and settle on a plan." Claude suggested moving from the 125M-parameter transformer to the 355M one. I did not accept this as an upgrade: the larger model scored 0.6101 against 0.6120 for the base model, 0.0019 lower or about 3 test rows, which is smaller than the variation between folds, and it needed 2.8 times the parameters and four times the training time. I still kept its predictions in the blend, because a model that is not better on its own can still make mistakes on different rows. I accepted the suggestion to train the base model with three random seeds and average them. The three runs scored 0.6015, 0.5995 and 0.6102, a spread of 0.0107, which shows how much noise one run carries; the average reached 0.6120. Two other suggestions were rejected after testing. Adding BERTweet to the blend changed it by only 0.0002, which is not a real result. Turning on BERTweet's emoji-to-text normaliser changed the out-of-fold score by −0.0018. The reason is worth recording: a byte-level tokenizer already has tokens for emoji, so emoji are only lost in the TF-IDF views, where the default token pattern drops them.

**Emotion-trained checkpoints.** *Prompt (13 November):* "Find a way to push the score higher. Is that hard?" Claude suggested starting from checkpoints that were already trained to recognise emotions, and keeping their classification heads. I tried it. Each of the two j-hartmann models beat the tweet model on its own, adding them raised the nested score by about 0.02, and this gave a public score of 0.6430. Not every follow-up worked, and I tested each one in separate script runs. A cardiffnlp base model trained for multi-label emotion detection scored only 0.6004, so I dropped it. Two more seeds for each j-hartmann model improved each model by 0.005 to 0.008 on its own, but moved the blend by only 0.0007, about one row, because a blend already averages out much of the seed noise; the same happened with the tweet base model. Four hyperparameter changes on the distilled model (maximum length 128, eight epochs, learning rate 5e-5, batch size 16) scored 0.6175, 0.6100, 0.6084 and 0.6172. All four are inside the range of 0.6162 to 0.6240 that the original settings give when only the seed changes, so I kept the original settings. Longer training and the higher learning rate were both a little worse, which fits a small dataset that overfits easily. The three other checkpoints in the seven-model blend added little on their own, but together they raised the nested score by 0.0049 in the script run (three folds up, two down, paired t = 1.18) and by 0.0098 in the notebook run. I submitted it and treated the gain as a trend. Two days later I removed these checkpoints from the final entries, and after the TAs replied I used them again in one of the two. The next paragraph and 1.3 explain why.

**Where the data came from, and how I used the emotion checkpoints.** *Prompts (17, 18 and 21 November):* "Keep optimising and submitting until the deadline.", later "I want a good score, but not by any means.", and finally "Read this file first, then guide me through the Kaggle submission step by step." On 17 November, Claude suggested checking whether public emotion datasets contain the competition texts. They do: 48.8% of the test texts are GoEmotions comments, and on the training set the top GoEmotions label matches the competition label 95.3% of the time. One option Claude raised was to use those public labels directly for the matching test rows. I rejected it, because that would be looking up the answers. Claude then suggested training on GoEmotions and SemEval-2018 after removing every competition text, first in two stages and then together with the competition data. The joint version gave my best single models (up to 0.6426 out of fold), and a blend that reached 0.6545 nested and first place on the public leaderboard (0.6539). Claude recommended submitting these blends and described the overlap with GoEmotions as a technical note, not a problem. I did not agree. The checkpoints in Section 3.6 were trained on GoEmotions with its labels, so in effect they had seen about half of my test answers. When I checked on the training set, their whole advantage came from the GoEmotions rows. My own GoEmotions-trained models showed the same pattern, and after the removal the closest text was still 0.698 similar to a competition text. At first I decided to use only models that learned from the competition data and from unlabelled text, and I asked the TAs about the overlap. They replied that it is fine as long as I train my own model with the data provided by the competition. After that reply, I used one final entry of each kind: the notebook's seven-model blend with the emotion checkpoints (Section 3.7) and the blend without them (Section 3.8). I did not use the blend that placed first on the public leaderboard at the time (0.6539), because its main models were trained by scripts outside this notebook. Two more variants of the external data (adding dair-ai tweets, and copying GoEmotions' one-vote labelling rule) did not help and were dropped as well.

**Using the test set without its labels.** *Prompt (14 November):* "Run the pseudo-labelling." Since every experiment pointed to the 3,360 training rows as the limit, Claude suggested pseudo-labelling: retrain with the test rows the blend is most sure about, using its own predictions as labels. The idea looked reasonable, because training rows with confidence of at least 0.8 were 94.0% correct, against about 62% overall. But the result got worse, and it got worse the more rows I added: 0.6041, 0.6035 and 0.6021 at thresholds of 0.9, 0.8 and 0.7, against 0.6053 for the same model without pseudo-labels. Each difference alone is inside the noise, but all three go in the same direction. The class mix explains why. Among the confident rows at 0.8, joy was 49.1% against 26.04% in the training data, and disgust was only 2.4% against 13.51%. The model is most sure exactly where it is already biased, so feeding those rows back tells it that the bias is right, and macro F1 punishes exactly that. I rejected it.

**The data itself.** *Prompt (11 November):* the first-submission prompt above. The most important suggestion here was to split the folds with `StratifiedGroupKFold`, grouping by the MD5 hash of the normalised text. I used it. The training set has 50 repeated texts covering 100 rows. Grouping by hash means no group spans more than one fold, so the model is never scored on rows it has already memorised, and the class shares stayed within 0.3 percentage points across the five folds. I also followed the suggestion to keep the 16 groups with conflicting labels instead of deleting them, because removing them would raise the validation score without making the model better on new text. At first I rejected the suggestion to give the 59 test texts that also appear in the training set their training labels. At that time my fine-tuned model already got 55 of the 59 right, so the rule would change only 4 of 1,440 rows and add a special case. I used the rule in my final entry without the checkpoints (Section 3.8). It only uses the training labels that were given to me, it skips the 16 groups with conflicting labels, and it changes 3 rows.

**How I judged whether something helped.** *Prompts (11 and 13 November):* the first-submission prompt and "Keep fine-tuning and settle on a plan." I followed the suggestion to freeze one five-fold split and reuse it everywhere. Each fold has about 672 rows and the class shares are within 0.3 percentage points, so every out-of-fold number in this report rests on the same basis, from the linear models to the transformers, and a difference of a few predictions can be compared directly. I also followed three related suggestions. Fitting one temperature per model before searching for blend weights was necessary: without it, the search put a weight of 0.85 on the weakest model only to make up for its flatter probabilities. Reporting the nested score instead of the direct one cost 0.0052 (0.6236 against 0.6288), so the direct score was 0.0052 too high. And the rule that a difference smaller than the between-fold standard deviation does not count is why I did not treat the class weighting as a real gain or the larger model as an upgrade, so this rule mattered.

**Submissions.** *Prompts (11, 12 and 14 November):* "Can this first submission be sent the same way as in my other Kaggle project, from PowerShell?", "Explain what the probe submission tells us." and "After a submission, shouldn't we look at the private score, not the public one?" I followed the suggestion to spend one submission on a probe: a file that predicts `joy` for all 1,440 rows, whose score could be worked out in advance for each possible metric. It returned 0.0686, which matches the macro F1 prediction of 0.0689 and rules out micro (0.2604) and weighted (0.1076), so one submission settled what the competition page did not say. I also produced both header versions, `id` and `post_id`, because the competition's own files disagreed about which one was expected; `id` was accepted. The most important suggestion in this group was to choose models by the out-of-fold score and use the public leaderboard only as a sanity check. The submission history shows why: the three calibrated blends rank 0.6204, 0.6216 and 0.6236 locally, and 0.6364, 0.6330 and 0.6223 on the public leaderboard, exactly the reverse order, on a 432-row sample where the whole spread is 6 predictions. The emotion blends did the same: the seven-model blend is 0.0071 higher out of fold than the four-model one, and 0.0064 lower in public. For the final pair I chose two submissions that this notebook produces, one with and one without the emotion checkpoints: the Section 3.7 seven-model file and the Section 3.8 file. I selected them by hand on 21 November. If I had let Kaggle choose, it would have picked the two best public scores, and one of them depends on models trained outside this notebook. Finally, every submission file is written by a function that runs five checks first, on column count, row count, label values, missing values and exact label spelling, so a broken file cannot reach Kaggle.

---

**`From here on starts the code section for the competition.`**

---

# **Competition Code**

## 1. Preprocessing Steps

In [1]:
### Add the code related to the preprocessing steps in cells inside this section

# T0 step 1 - load the three competition files WITHOUT assuming any column name.
# The Kaggle Data page calls the label column "emotion" but the real header is "label",
# so every column list is printed and the printed names are what the rest of the code uses.
import json, re, time, hashlib, warnings
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_STATE = 42          # every randomised step reuses this one seed
N_SPLITS = 5               # one fixed 5-fold split is shared by every experiment
DATA_DIR = Path("data/kaggle")                 # competition files (kept out of git)
OUT_DIR = Path("../_workbench/kaggle")         # experiment records, never committed
for sub in ["experiments", "oof", "submissions"]:
    (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)

with open(DATA_DIR / "final_posts.json", encoding="utf-8") as fh:
    raw_posts = json.load(fh)
print("final_posts.json is a", type(raw_posts).__name__, "of length", len(raw_posts))
print("keys of the first record:", list(raw_posts[0].keys()))

posts = pd.DataFrame(raw_posts)
identification = pd.read_csv(DATA_DIR / "data_identification.csv")
emotion = pd.read_csv(DATA_DIR / "emotion.csv")
sample_submission = pd.read_csv(DATA_DIR / "sampleSubmission.csv")

for name, frame in [("posts", posts), ("identification", identification),
                    ("emotion", emotion), ("sample_submission", sample_submission)]:
    print(f"{name:18s} shape={str(frame.shape):12s} columns={list(frame.columns)}")

print("\nposts.post_id is unique:", posts["post_id"].is_unique)
print("missing values in posts:", posts.isna().sum().to_dict())
print("train/test split recorded in data_identification.csv:",
      identification["set"].value_counts().to_dict())


final_posts.json is a list of length 4800
keys of the first record: ['post_id', 'text']
posts              shape=(4800, 2)    columns=['post_id', 'text']
identification     shape=(4800, 2)    columns=['post_id', 'set']
emotion            shape=(3360, 2)    columns=['post_id', 'label']
sample_submission  shape=(1440, 2)    columns=['post_id', 'label']

posts.post_id is unique: True
missing values in posts: {'post_id': 0, 'text': 0}
train/test split recorded in data_identification.csv: {'train': 3360, 'test': 1440}


In [2]:
# T0 step 2 - merge the three files on post_id and assert the structure we were promised.
# A silent merge failure here would poison everything downstream, so each expectation
# is checked with an assert rather than just printed and eyeballed.
merged = posts.merge(identification, on="post_id", how="outer", indicator=True)
print("merge coverage posts vs identification:", merged["_merge"].value_counts().to_dict())
assert (merged["_merge"] == "both").all(), "some post_id appears in only one of the two files"

merged = merged.drop(columns="_merge").merge(emotion, on="post_id", how="left")
train = merged[merged["set"] == "train"].reset_index(drop=True)
test = merged[merged["set"] == "test"].reset_index(drop=True)

print(f"\ntrain rows {len(train)} + test rows {len(test)} = {len(train) + len(test)}")
assert len(train) + len(test) == 4800, "train + test should cover all 4,800 posts"
assert train["label"].notna().all(), "every training row must carry a label"
assert test["label"].isna().all(), "no test row may carry a label"
assert len(test) == len(sample_submission), "test rows must match sampleSubmission rows"
assert set(test["post_id"]) == set(sample_submission["post_id"]), "test ids must match the sample file"
print("test ids match sampleSubmission as a set:", True)
print("test ids are in the same ORDER as sampleSubmission:",
      list(test["post_id"]) == list(sample_submission["post_id"]))
print("-> the submission must therefore be re-ordered to the sample file, not written in test order")

LABELS = sorted(train["label"].unique())
print("\nthe six labels, exactly as spelled in emotion.csv:", LABELS)


merge coverage posts vs identification: {'both': 4800, 'left_only': 0, 'right_only': 0}

train rows 3360 + test rows 1440 = 4800
test ids match sampleSubmission as a set: True
test ids are in the same ORDER as sampleSubmission: False
-> the submission must therefore be re-ordered to the sample file, not written in test order

the six labels, exactly as spelled in emotion.csv: ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise']


In [3]:
# T0 step 3 - describe the data, and turn the class distribution into the three numbers
# that the probe submission will be read against.
# Mean F1 is not defined anywhere on the competition pages (macro / micro / weighted all
# fit the words), so one throw-away submission that predicts a single class everywhere
# will tell us which one it is. If the largest class covers a share p of the scored subset,
# that submission gets precision = p, recall = 1, so its F1 for that class is 2p/(1+p),
# and the three candidate metrics give three very different totals.
counts = train["label"].value_counts()
distribution = pd.DataFrame({
    "n": counts,
    "share_%": (counts / len(train) * 100).round(2),
})
print("=== class distribution in the training set ===")
print(distribution.to_string())
print("\nsmallest class:", counts.idxmin(), counts.min(),
      "| largest class:", counts.idxmax(), counts.max(),
      f"| imbalance ratio {counts.max() / counts.min():.2f}x")

MAJORITY_LABEL = counts.idxmax()
p = counts.max() / len(train)
f1_of_that_class = 2 * p / (1 + p)
probe_expectation = {
    "macro": f1_of_that_class / 6,
    "micro": p,
    "weighted": p * f1_of_that_class,
}
print(f"\n=== what a constant '{MAJORITY_LABEL}' submission should score (p = {p:.4f}) ===")
for metric, value in probe_expectation.items():
    print(f"  if Mean F1 means {metric:9s} -> {value:.4f}")
print(f"  weighted / macro = {probe_expectation['weighted'] / probe_expectation['macro']:.2f}x,"
      " far enough apart to decide from a single submission")
print("  caveat: p is estimated from the training split; the public subset is only 30% of the"
      " test set and its true class shares are unknown.")

train_lengths = pd.DataFrame({
    "chars": train["text"].str.len(),
    "words": train["text"].str.split().str.len(),
})
print("\n=== text length, training set ===")
print(train_lengths.describe().round(2).to_string())
print("\n=== words per post, by class ===")
print(train.assign(words=train_lengths["words"])
          .groupby("label")["words"].agg(["mean", "median", "max"]).round(2).to_string())
print("\n-> the length distribution is heavily right-skewed (median 13 words, maximum above 500),"
      " which is why TF-IDF is used with sublinear term frequencies below.")


=== class distribution in the training set ===
            n  share_%
label                 
joy       875    26.04
anger     674    20.06
sadness   628    18.69
disgust   454    13.51
surprise  441    13.12
fear      288     8.57

smallest class: fear 288 | largest class: joy 875 | imbalance ratio 3.04x

=== what a constant 'joy' submission should score (p = 0.2604) ===
  if Mean F1 means macro     -> 0.0689
  if Mean F1 means micro     -> 0.2604
  if Mean F1 means weighted  -> 0.1076
  weighted / macro = 1.56x, far enough apart to decide from a single submission
  caveat: p is estimated from the training split; the public subset is only 30% of the test set and its true class shares are unknown.

=== text length, training set ===
         chars    words
count  3360.00  3360.00
mean     87.78    16.51
std     121.88    24.96
min       5.00     1.00
25%      42.00     8.00
50%      68.00    13.00
75%     101.25    19.00
max    2545.00   508.00

=== words per post, by class ===
         

In [4]:
# T0 step 4 - build the ONE fixed cross-validation split that every later experiment reuses.
# The four steps below must happen in this order:
#   1. normalise the text (lowercase, collapse whitespace)
#   2. count duplicate texts
#   3. if duplicates exist, group them so that copies of one text cannot be split across folds
#   4. only then write folds.csv
# Skipping step 3 would let the same sentence sit in both the training and the validation
# fold, which inflates every out-of-fold score and makes all later comparisons worthless.
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold

def normalise(text):
    """Lowercase and collapse runs of whitespace - used only for duplicate detection."""
    return re.sub(r"\s+", " ", str(text).lower()).strip()

train["text_normalised"] = train["text"].map(normalise)
test["text_normalised"] = test["text"].map(normalise)

train_dupes = train["text_normalised"].value_counts()
train_dupes = train_dupes[train_dupes > 1]
test_dupes = test["text_normalised"].value_counts()
test_dupes = test_dupes[test_dupes > 1]
label_disagreement = train.groupby("text_normalised")["label"].nunique()

print("=== duplicate texts after normalisation ===")
print(f"train: {len(train_dupes)} distinct texts appear more than once, covering"
      f" {int(train_dupes.sum())} rows (largest group {int(train_dupes.max())} rows)")
print(f"test : {len(test_dupes)} distinct texts appear more than once, covering"
      f" {int(test_dupes.sum())} rows")
print(f"identical training texts carrying DIFFERENT labels: {(label_disagreement > 1).sum()}")
print("-> those contradictory pairs put a ceiling on any achievable score, and they are the"
      " reason the fold split has to be grouped.")

overlap = set(train["text_normalised"]) & set(test["text_normalised"])
print(f"\ntexts that occur in BOTH train and test: {len(overlap)} distinct"
      f" ({train['text_normalised'].isin(overlap).sum()} train rows,"
      f" {test['text_normalised'].isin(overlap).sum()} test rows)")
print("-> recorded as a limitation in the report: part of the test set is memorisable.")

train["group"] = train["text_normalised"].map(
    lambda s: hashlib.md5(s.encode("utf-8")).hexdigest()
)
if len(train_dupes) > 0:
    splitter = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    split_iter = splitter.split(train, train["label"], groups=train["group"])
    print("\nduplicates found -> using StratifiedGroupKFold with the text hash as the group")
else:
    splitter = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    split_iter = splitter.split(train, train["label"])
    print("\nno duplicates -> plain StratifiedKFold is enough")

train["fold"] = -1
for fold_id, (_, validation_index) in enumerate(split_iter):
    train.loc[validation_index, "fold"] = fold_id
assert (train["fold"] >= 0).all(), "every training row must land in exactly one fold"

print("\nrows per fold:", train["fold"].value_counts().sort_index().to_dict())
print("groups that leaked across more than one fold:",
      int((train.groupby("group")["fold"].nunique() > 1).sum()), "(must be 0)")
print("\nclass share per fold, in % (stratification check):")
print((pd.crosstab(train["fold"], train["label"], normalize="index") * 100).round(2).to_string())

train[["post_id", "label", "fold"]].to_csv(OUT_DIR / "folds.csv", index=False)
print(f"\nwritten: {OUT_DIR / 'folds.csv'} - this file is now frozen for the whole competition")


=== duplicate texts after normalisation ===
train: 50 distinct texts appear more than once, covering 100 rows (largest group 2 rows)
test : 8 distinct texts appear more than once, covering 16 rows
identical training texts carrying DIFFERENT labels: 16
-> those contradictory pairs put a ceiling on any achievable score, and they are the reason the fold split has to be grouped.

texts that occur in BOTH train and test: 59 distinct (61 train rows, 59 test rows)
-> recorded as a limitation in the report: part of the test set is memorisable.

duplicates found -> using StratifiedGroupKFold with the text hash as the group

rows per fold: {0: 673, 1: 672, 2: 672, 3: 672, 4: 671}
groups that leaked across more than one fold: 0 (must be 0)

class share per fold, in % (stratification check):
label  anger  disgust  fear    joy  sadness  surprise
fold                                                 
0      20.06    13.52  8.62  26.00    18.72     13.08
1      20.09    13.54  8.48  26.04    18.75    

## 2. Feature Engineering Steps

In [5]:
### Add the code related to the feature engineering steps in cells inside this section

# Feature family 1 and 2 - two TF-IDF views of the same text, horizontally stacked.
#   word level  (1,2)-grams: catches vocabulary and short phrases such as "not happy"
#   char level  char_wb (2,5)-grams: catches lengthened words ("angryyy"), misspellings and
#               emoji, which the word view throws away as unseen tokens
# Both vectorisers are fitted INSIDE each fold on the training part only. Fitting them once
# on all 3,360 rows would let validation-fold vocabulary and document frequencies leak into
# the features and would quietly inflate every score.
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer

WORD_TFIDF = dict(ngram_range=(1, 2), min_df=2, sublinear_tf=True, strip_accents="unicode")
CHAR_TFIDF = dict(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)

# A one-off look at how wide each view is when fitted on four of the five folds.
demo_train_text = train.loc[train["fold"] != 0, "text"].values
for view_name, settings in [("word", WORD_TFIDF), ("char_wb", CHAR_TFIDF)]:
    matrix = TfidfVectorizer(**settings).fit_transform(demo_train_text)
    density = matrix.nnz / (matrix.shape[0] * matrix.shape[1]) * 100
    print(f"{view_name:8s} view: {matrix.shape[1]:6d} features, {density:.3f}% of entries non-zero")
print("\n-> the character view is roughly four times wider than the word view and is the"
      " cheapest single improvement available at this stage.")


word     view:   6107 features, 0.263% of entries non-zero
char_wb  view:  24566 features, 0.755% of entries non-zero

-> the character view is roughly four times wider than the word view and is the cheapest single improvement available at this stage.


In [6]:
# Feature family 3 - MiniLM sentence embeddings.
# This is the same all-MiniLM-L6-v2 model used in Master section 5, so no new download is
# needed. Unlike TF-IDF it carries meaning learned from a large external corpus, so two posts
# can be close even with no shared words; that is exactly what the sparse views cannot do.
# The encoder is UNSUPERVISED with respect to our labels, so encoding all 4,800 texts once -
# outside the fold loop - leaks nothing, and it keeps the cross-validation cheap.
from sentence_transformers import SentenceTransformer

EMBEDDING_CACHE = OUT_DIR / "minilm_embeddings.npz"
if EMBEDDING_CACHE.exists():
    cached = np.load(EMBEDDING_CACHE)
    train_embeddings, test_embeddings = cached["train"], cached["test"]
    print("loaded cached embeddings from", EMBEDDING_CACHE)
else:
    encoder = SentenceTransformer("all-MiniLM-L6-v2")
    started = time.time()
    train_embeddings = encoder.encode(list(train["text"]), batch_size=64,
                                      normalize_embeddings=True, show_progress_bar=False)
    test_embeddings = encoder.encode(list(test["text"]), batch_size=64,
                                     normalize_embeddings=True, show_progress_bar=False)
    np.savez(EMBEDDING_CACHE, train=train_embeddings, test=test_embeddings)
    print(f"encoded {len(train) + len(test)} texts in {time.time() - started:.0f}s"
          f" and cached them to {EMBEDDING_CACHE}")

print("train embeddings", train_embeddings.shape, "| test embeddings", test_embeddings.shape)
print("rows are L2-normalised:", np.allclose(np.linalg.norm(train_embeddings, axis=1), 1.0))

EMBEDDING_SCALE = 1.0   # embeddings sit beside TF-IDF columns; this keeps the two blocks comparable

def build_features(train_text, apply_text, train_embedding, apply_embedding, views):
    """Fit the requested views on train_text only, then apply them to apply_text."""
    fitted_train, fitted_apply = [], []
    if "word" in views:
        vectoriser = TfidfVectorizer(**WORD_TFIDF)
        fitted_train.append(vectoriser.fit_transform(train_text))
        fitted_apply.append(vectoriser.transform(apply_text))
    if "char" in views:
        vectoriser = TfidfVectorizer(**CHAR_TFIDF)
        fitted_train.append(vectoriser.fit_transform(train_text))
        fitted_apply.append(vectoriser.transform(apply_text))
    if "emb" in views:
        fitted_train.append(csr_matrix(train_embedding * EMBEDDING_SCALE))
        fitted_apply.append(csr_matrix(apply_embedding * EMBEDDING_SCALE))
    if len(fitted_train) == 1:
        return fitted_train[0], fitted_apply[0]
    return hstack(fitted_train).tocsr(), hstack(fitted_apply).tocsr()

print("\nbuild_features() is the single place where features are made, so the"
      " cross-validation below and the final full-data fit cannot drift apart.")


C:\MyProjects\DM\Lab-2\kaggle_gpu\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


loaded cached embeddings from ..\_workbench\kaggle\minilm_embeddings.npz
train embeddings (3360, 384) | test embeddings (1440, 384)
rows are L2-normalised: True

build_features() is the single place where features are made, so the cross-validation below and the final full-data fit cannot drift apart.


## 3. Model Implementation Steps

In [7]:
### Add the code related to the model implementation steps in cells inside this section

# Model search - out-of-fold macro F1 over the frozen 5-fold split from section 1.
# Every configuration is scored on exactly the same folds, so the numbers are comparable and
# the out-of-fold predictions could later be reused for an ensemble.
# Three things are varied, one at a time: which feature views are used, whether the classes
# are reweighted, and the regularisation strength C.
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, classification_report, confusion_matrix

warnings.filterwarnings("ignore", category=UserWarning)

y_train = train["label"].values
fold_ids = train["fold"].values
texts = train["text"].values

def evaluate(views, class_weight, C):
    """Return out-of-fold predictions and the three candidate F1 averages."""
    out_of_fold = np.empty(len(train), dtype=object)
    per_fold = []
    started = time.time()
    for fold_id in range(N_SPLITS):
        is_train = fold_ids != fold_id
        is_validation = fold_ids == fold_id
        X_train, X_validation = build_features(
            texts[is_train], texts[is_validation],
            train_embeddings[is_train], train_embeddings[is_validation], views,
        )
        model = LinearSVC(C=C, class_weight=class_weight,
                          random_state=RANDOM_STATE, max_iter=10000)
        model.fit(X_train, y_train[is_train])
        predictions = model.predict(X_validation)
        out_of_fold[is_validation] = predictions
        per_fold.append(f1_score(y_train[is_validation], predictions, average="macro"))
    return out_of_fold, {
        "views": "+".join(views),
        "class_weight": str(class_weight),
        "C": C,
        "oof_macro_f1": f1_score(y_train, out_of_fold, average="macro"),
        "oof_micro_f1": f1_score(y_train, out_of_fold, average="micro"),
        "oof_weighted_f1": f1_score(y_train, out_of_fold, average="weighted"),
        "fold_mean": float(np.mean(per_fold)),
        "fold_std": float(np.std(per_fold)),
        "seconds": round(time.time() - started, 1),
    }

CANDIDATES = [
    (["word"],                "balanced", 0.10),   # single sparse view, for reference
    (["char"],                "balanced", 0.10),   # characters alone already beat words
    (["emb"],                 "balanced", 1.00),   # dense view alone
    (["word", "char"],        None,       0.10),   # does reweighting help?
    (["word", "char"],        "balanced", 0.10),
    (["word", "char"],        "balanced", 0.20),   # does less regularisation help?
    (["word", "char", "emb"], "balanced", 0.05),
    (["word", "char", "emb"], "balanced", 0.10),
]

records, oof_store = [], {}
for views, class_weight, C in CANDIDATES:
    predictions, record = evaluate(views, class_weight, C)
    key = f"{record['views']}|cw={record['class_weight']}|C={C}"
    oof_store[key] = predictions
    records.append(record)
    print(f"{key:40s} macro={record['oof_macro_f1']:.4f} (folds "
          f"{record['fold_mean']:.4f} +/- {record['fold_std']:.4f})  "
          f"micro={record['oof_micro_f1']:.4f}  weighted={record['oof_weighted_f1']:.4f}  "
          f"{record['seconds']}s", flush=True)

results = pd.DataFrame(records).sort_values("oof_macro_f1", ascending=False)
results.to_csv(OUT_DIR / "experiments" / "t1_grid.csv", index=False)
print("\n=== ranked by out-of-fold macro F1 ===")
print(results[["views", "class_weight", "C", "oof_macro_f1", "fold_std",
               "oof_micro_f1", "oof_weighted_f1"]].to_string(index=False))

best = results.iloc[0]
print(f"\nbest configuration: views={best['views']}, class_weight={best['class_weight']},"
      f" C={best['C']} -> out-of-fold macro F1 {best['oof_macro_f1']:.4f}"
      f" (fold standard deviation {best['fold_std']:.4f})")
print("the top configurations sit within one fold standard deviation of each other, so the"
      " ranking inside that group is noise; what is NOT noise is the gap opened by adding"
      " the embedding view.")


word|cw=balanced|C=0.1                   macro=0.4730 (folds 0.4728 +/- 0.0134)  micro=0.4940  weighted=0.4915  0.4s
char|cw=balanced|C=0.1                   macro=0.5045 (folds 0.5041 +/- 0.0116)  micro=0.5229  weighted=0.5201  1.7s
emb|cw=balanced|C=1.0                    macro=0.4863 (folds 0.4856 +/- 0.0091)  micro=0.5071  weighted=0.5075  1.9s
word+char|cw=None|C=0.1                  macro=0.5096 (folds 0.5086 +/- 0.0188)  micro=0.5357  weighted=0.5248  1.9s
word+char|cw=balanced|C=0.1              macro=0.5129 (folds 0.5122 +/- 0.0133)  micro=0.5280  weighted=0.5264  1.9s
word+char|cw=balanced|C=0.2              macro=0.5146 (folds 0.5138 +/- 0.0159)  micro=0.5307  weighted=0.5292  2.3s
word+char+emb|cw=balanced|C=0.05         macro=0.5708 (folds 0.5705 +/- 0.0167)  micro=0.5878  weighted=0.5851  3.6s
word+char+emb|cw=balanced|C=0.1          macro=0.5657 (folds 0.5654 +/- 0.0154)  micro=0.5839  weighted=0.5819  3.5s

=== ranked by out-of-fold macro F1 ===
        views class_weig

In [8]:
# Refit the chosen configuration on all 3,360 labelled rows and predict the test set.
# The vectorisers are fitted here on the full training text - that is correct, because the
# test labels are never touched; only the fold-level fits had to stay inside the fold.
BEST_VIEWS = ["word", "char", "emb"]
BEST_CLASS_WEIGHT = "balanced"
BEST_C = 0.05

X_full, X_test = build_features(train["text"].values, test["text"].values,
                                train_embeddings, test_embeddings, BEST_VIEWS)
print("full training matrix", X_full.shape, "| test matrix", X_test.shape)

final_model = LinearSVC(C=BEST_C, class_weight=BEST_CLASS_WEIGHT,
                        random_state=RANDOM_STATE, max_iter=10000)
final_model.fit(X_full, y_train)
test_predictions = final_model.predict(X_test)

print("\npredicted class distribution on the test set, next to the training distribution:")
comparison = pd.DataFrame({
    "test_predicted_%": (pd.Series(test_predictions).value_counts(normalize=True) * 100).round(2),
    "train_actual_%": (train["label"].value_counts(normalize=True) * 100).round(2),
})
print(comparison.to_string())
print("\n-> class_weight='balanced' pushes the rare classes up, so the predicted shares are"
      " flatter than the training shares. That is deliberate: macro F1 rewards the small"
      " classes as much as the large ones.")

# Keep the out-of-fold predictions of the chosen configuration for the report and for any
# later ensemble work.
best_key = f"word+char+emb|cw={BEST_CLASS_WEIGHT}|C={BEST_C}"
oof_frame = pd.DataFrame({
    "post_id": train["post_id"], "true_label": y_train,
    "oof_prediction": oof_store[best_key], "fold": fold_ids,
})
oof_frame.to_csv(OUT_DIR / "oof" / "tfidf_char_minilm_linearsvc.csv", index=False)

print("\n=== per-class out-of-fold performance of the chosen configuration ===")
print(classification_report(y_train, oof_store[best_key], digits=3))
print("confusion matrix (rows = true label, columns = predicted):")
print(pd.DataFrame(confusion_matrix(y_train, oof_store[best_key], labels=LABELS),
                   index=LABELS, columns=LABELS).to_string())


full training matrix (3360, 35420) | test matrix (1440, 35420)

predicted class distribution on the test set, next to the training distribution:
          test_predicted_%  train_actual_%
anger                22.01           20.06
disgust              10.83           13.51
fear                  8.61            8.57
joy                  28.68           26.04
sadness              16.88           18.69
surprise             12.99           13.12

-> class_weight='balanced' pushes the rare classes up, so the predicted shares are flatter than the training shares. That is deliberate: macro F1 rewards the small classes as much as the large ones.

=== per-class out-of-fold performance of the chosen configuration ===
              precision    recall  f1-score   support

       anger      0.545     0.580     0.562       674
     disgust      0.507     0.421     0.460       454
        fear      0.609     0.580     0.594       288
         joy      0.668     0.727     0.696       875
     sadness

In [9]:
# Write the submission files, and refuse to write anything that fails the five checks.
# Two header spellings are produced on purpose: the Overview page and the Kaggle API both
# say the identifier column is "id", while the downloaded sampleSubmission.csv header says
# "post_id". The probe submission is uploaded first and settles which one Kaggle accepts.
SUBMISSION_DIR = OUT_DIR / "submissions"

def check_and_write(post_ids, labels, stem):
    """Run the five pre-submission checks, then write the id and post_id header variants."""
    frame = pd.DataFrame({"post_id": post_ids, "label": labels})
    # the scored file must follow sampleSubmission's row order, so join onto it
    frame = sample_submission[["post_id"]].merge(frame, on="post_id", how="left")

    problems = []
    if list(frame.columns) != ["post_id", "label"]:
        problems.append(f"1. expected exactly two columns, got {list(frame.columns)}")
    if len(frame) != len(sample_submission):
        problems.append(f"2. expected {len(sample_submission)} rows, got {len(frame)}")
    unexpected = set(frame["label"].dropna()) - set(LABELS)
    if unexpected:
        problems.append(f"3. labels outside the six classes: {unexpected}")
    if frame.isna().any().any():
        problems.append(f"4. missing values: {frame.isna().sum().to_dict()}")
    if not set(frame["label"].dropna()) <= set(emotion["label"].unique()):
        problems.append("5. label strings do not match emotion.csv exactly")
    if problems:
        raise ValueError("submission rejected:\n  " + "\n  ".join(problems))

    written = []
    for id_column in ["id", "post_id"]:
        out = frame.rename(columns={"post_id": id_column})
        path = SUBMISSION_DIR / f"{stem}__{id_column}.csv"
        out.to_csv(path, index=False)
        written.append(path)
    print(f"all five checks passed for '{stem}' "
          f"({len(frame)} rows, {frame['label'].nunique()} distinct labels)")
    for path in written:
        print("   written:", path)
    return frame

# Submission A - the probe. Predicts the majority class everywhere, purely to read off which
# average "Mean F1" refers to. It must never be chosen as a final submission.
probe = check_and_write(test["post_id"].values,
                        np.full(len(test), MAJORITY_LABEL), "probe_constant")
print(f"\nPROBE: every row predicted '{MAJORITY_LABEL}'. Expected score if the metric is"
      f" macro {probe_expectation['macro']:.4f} / micro {probe_expectation['micro']:.4f}"
      f" / weighted {probe_expectation['weighted']:.4f}")

# Submission B - the real first entry.
submission = check_and_write(test["post_id"].values, test_predictions, "t1_tfidf_char_minilm")
print(f"\nREAL ENTRY: out-of-fold macro F1 {best['oof_macro_f1']:.4f};"
      " the first three rows of the file are")
print(submission.head(3).to_string(index=False))

print("\nUpload order: the probe first (it settles both the metric and the header), then the"
      " real entry. Both go in the submission ledger before anything is uploaded.")


all five checks passed for 'probe_constant' (1440 rows, 1 distinct labels)
   written: ..\_workbench\kaggle\submissions\probe_constant__id.csv
   written: ..\_workbench\kaggle\submissions\probe_constant__post_id.csv

PROBE: every row predicted 'joy'. Expected score if the metric is macro 0.0689 / micro 0.2604 / weighted 0.1076
all five checks passed for 't1_tfidf_char_minilm' (1440 rows, 6 distinct labels)
   written: ..\_workbench\kaggle\submissions\t1_tfidf_char_minilm__id.csv
   written: ..\_workbench\kaggle\submissions\t1_tfidf_char_minilm__post_id.csv

REAL ENTRY: out-of-fold macro F1 0.5708; the first three rows of the file are
post_id   label
xyzx9jp   anger
l3vzzxt sadness
g680iug sadness

Upload order: the probe first (it settles both the metric and the header), then the real entry. Both go in the submission ledger before anything is uploaded.


### 3.4 Fine-tuning a tweet-domain transformer

The models above all take a fixed representation and fit a classifier on top of it. This section changes the representation itself: a transformer pre-trained on 154 million tweets, fine-tuned end to end on our 3,360 labelled ones, on the same frozen folds so the score is comparable with everything above.

**This section needs the GPU kernel** (`Python (lab2-kaggle-gpu)`). Five folds take about two minutes on an RTX 5090; the whole section, including three seeds and the larger model, takes roughly fifteen.

In [10]:
# Fine-tuning helpers. This section needs the GPU kernel (Python (lab2-kaggle-gpu)):
# the linear models above are seconds of CPU work, but fine-tuning a transformer five
# times is not. Everything else about the experiment is unchanged - the same frozen
# folds, the same macro F1, the same comparison - so these numbers sit beside the
# linear ones in the table above rather than replacing them.
import time

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT AVAILABLE")

LABEL_TO_INDEX = {label: i for i, label in enumerate(LABELS)}
y_index = np.array([LABEL_TO_INDEX[label] for label in y_train])


def tweet_normalise(text):
    """What the cardiffnlp tweet models were pre-trained on: handles and links masked.
    Emoji are left alone on purpose - a byte-level tokenizer has tokens for them, so
    unlike the TF-IDF views above this representation can actually see them."""
    text = re.sub(r"@\w+", "@user", str(text))
    text = re.sub(r"http\S+", "http", text)
    return " ".join(text.split())


class TweetDataset(Dataset):
    """Tokenise once, at a fixed length, so every batch has the same shape."""

    def __init__(self, texts, labels, tokeniser, max_len):
        self.encoded = tokeniser([tweet_normalise(t) for t in texts], truncation=True,
                                 max_length=max_len, padding="max_length", return_tensors="pt")
        self.labels = None if labels is None else torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return self.encoded["input_ids"].shape[0]

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.encoded.items()}
        if self.labels is not None:
            item["labels"] = self.labels[i]
        return item


@torch.no_grad()
def predict_logits(model, loader):
    model.eval()
    out = []
    for batch in loader:
        batch = {k: v.cuda() for k, v in batch.items() if k != "labels"}
        with torch.autocast("cuda", dtype=torch.bfloat16):
            out.append(model(**batch).logits.float().cpu())
    return torch.cat(out).numpy()


def finetune_cv(model_name, seed=RANDOM_STATE, epochs=5, lr=3e-5, batch_size=32, max_len=96):
    """Fine-tune model_name once per fold of the frozen split.

    Returns out-of-fold logits for the 3,360 training rows and the average of the five
    folds' logits for the 1,440 test rows. Averaging the folds is a small ensemble for
    free: five models trained on overlapping four-fifths of the data, which is steadier
    than any one of them.
    """
    torch.manual_seed(seed)
    np.random.seed(seed)
    oof = np.zeros((len(train), len(LABELS)), dtype=np.float32)
    test_logits = np.zeros((len(test), len(LABELS)), dtype=np.float32)
    started = time.time()

    for fold in range(N_SPLITS):
        is_validation = fold_ids == fold
        tokeniser = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=len(LABELS), ignore_mismatched_sizes=True).cuda()

        loader = DataLoader(TweetDataset(texts[~is_validation], y_index[~is_validation],
                                         tokeniser, max_len),
                            batch_size=batch_size, shuffle=True)
        optimiser = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
        schedule = torch.optim.lr_scheduler.OneCycleLR(
            optimiser, max_lr=lr, total_steps=len(loader) * epochs, pct_start=0.1)

        model.train()
        for _ in range(epochs):
            for batch in loader:
                batch = {k: v.cuda() for k, v in batch.items()}
                with torch.autocast("cuda", dtype=torch.bfloat16):
                    logits = model(**{k: v for k, v in batch.items() if k != "labels"}).logits
                    loss = F.cross_entropy(logits, batch["labels"])
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimiser.step()
                schedule.step()
                optimiser.zero_grad(set_to_none=True)

        oof[is_validation] = predict_logits(
            model, DataLoader(TweetDataset(texts[is_validation], None, tokeniser, max_len),
                              batch_size=128))
        test_logits += predict_logits(
            model, DataLoader(TweetDataset(test["text"].values, None, tokeniser, max_len),
                              batch_size=128)) / N_SPLITS
        print(f"  fold {fold}: macro F1 "
              f"{f1_score(y_index[is_validation], oof[is_validation].argmax(1), average='macro'):.4f}",
              flush=True)
        del model
        torch.cuda.empty_cache()

    score = f1_score(y_index, oof.argmax(1), average="macro")
    print(f"  out-of-fold macro F1 {score:.4f}  ({time.time() - started:.0f}s)", flush=True)
    return oof, test_logits, score


GPU: NVIDIA GeForce RTX 5090 Laptop GPU


In [11]:
# The tweet-domain base model, trained three times with different seeds.
#
# Why a tweet model rather than a general one: the same argument the Master notebook made
# for MiniLM in Section 5. Our corpus is 3,360 tweets, which is far too little to learn
# what words mean; this checkpoint was pre-trained on 154 million of them.
#
# Why three seeds: a single fine-tuning run carries its own randomness - the classifier
# head starts from scratch and the batches are shuffled. Averaging three runs cancels
# part of that, and the spread between them measures how much it was worth cancelling.
BASE_MODEL = "cardiffnlp/twitter-roberta-base-2022-154m"
SEEDS = [RANDOM_STATE, 7, 2026]

base_oof, base_test, base_scores = [], [], []
for seed in SEEDS:
    print(f"{BASE_MODEL}  seed={seed}")
    oof, test_logits, score = finetune_cv(BASE_MODEL, seed=seed)
    base_oof.append(oof)
    base_test.append(test_logits)
    base_scores.append(score)

base_avg_oof = np.mean(base_oof, axis=0)
base_avg_test = np.mean(base_test, axis=0)
base_avg_score = f1_score(y_index, base_avg_oof.argmax(1), average="macro")

print(f"\nthe three seeds on their own : {[round(s, 4) for s in base_scores]}"
      f"  (spread {max(base_scores) - min(base_scores):.4f})")
print(f"averaged over the three seeds: {base_avg_score:.4f}"
      f"  ({base_avg_score - max(base_scores):+.4f} on the best single seed)")
print(f"the best linear model above  : {best['oof_macro_f1']:.4f}"
      f"  -> fine-tuning gains {base_avg_score - best['oof_macro_f1']:+.4f},"
      f" about {(base_avg_score - best['oof_macro_f1']) * len(test):.0f} of the {len(test)} test rows")


cardiffnlp/twitter-roberta-base-2022-154m  seed=42


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 32830.49it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-2022-154m
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  fold 0: macro F1 0.5988


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 28151.61it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-2022-154m
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  fold 1: macro F1 0.6066


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 24625.32it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-2022-154m
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


... (21 outputs left out)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 28135.31it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-2022-154m
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  fold 3: macro F1 0.6249


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 24620.91it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-2022-154m
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  fold 4: macro F1 0.5915
  out-of-fold macro F1 0.6102  (188s)

the three seeds on their own : [0.6015, 0.5995, 0.6102]  (spread 0.0107)
averaged over the three seeds: 0.6120  (+0.0018 on the best single seed)
the best linear model above  : 0.5708  -> fine-tuning gains +0.0412, about 59 of the 1440 test rows


In [12]:
# The same architecture at 355 million parameters instead of 125 million, to find out
# whether capacity is what this problem is short of. Section 7 of the Master notebook
# asked the same question with a 544,964-weight network against a 16,764-weight linear
# SVM and the answer was no; this is that question asked again at a scale where the
# model is not learning the language from scratch.
LARGE_MODEL = "cardiffnlp/twitter-roberta-large-2022-154m"

print(LARGE_MODEL)
large_oof, large_test, large_score = finetune_cv(LARGE_MODEL, epochs=5, lr=1.5e-5, batch_size=16)

print(f"\nlarge  {large_score:.4f}   base (seed-averaged) {base_avg_score:.4f}"
      f"   difference {large_score - base_avg_score:+.4f}"
      f"  = {(large_score - base_avg_score) * len(test):+.0f} of {len(test)} test rows")
print("2.8x the parameters and 4x the training time; read the difference against the"
      " fold-to-fold spread before calling it an improvement.")


cardiffnlp/twitter-roberta-large-2022-154m


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 29919.21it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-large-2022-154m
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  fold 0: macro F1 0.6098


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 29903.86it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-large-2022-154m
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  fold 1: macro F1 0.6068


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 40925.68it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-large-2022-154m
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  fold 2: macro F1 0.6340


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 43225.36it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-large-2022-154m
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  fold 3: macro F1 0.6188


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 31100.90it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-large-2022-154m
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  fold 4: macro F1 0.5796
  out-of-fold macro F1 0.6101  (758s)

large  0.6101   base (seed-averaged) 0.6120   difference -0.0018  = -3 of 1440 test rows
2.8x the parameters and 4x the training time; read the difference against the fold-to-fold spread before calling it an improvement.


### 3.5 Calibrating and blending the three models

Three models that score similarly can still be worth combining, because what matters is whether they are wrong about the *same* rows. The linear model reads surface words, the transformers read context, and the blend below is chosen on out-of-fold macro F1 rather than on the leaderboard.

In [13]:
# Blending the three models, and the two things that have to be got right first.
#
# (1) Scale. The linear model returns SVM margins, the transformers return logits, and
#     the two live on different scales. Pushed through softmax as they are, the linear
#     model looks nearly uniform - unconfident - and a weight search then spends its
#     freedom compensating for that rather than deciding which model to trust. A first
#     attempt without this step put 0.85 on the weakest of the three. Fitting one
#     temperature per model on out-of-fold log loss removes the scale difference, after
#     which the weights mean what they look like.
#
# (2) Honesty. Weights chosen on the out-of-fold predictions and then scored on those
#     same predictions are optimistic. The nested loop below picks the weights on four
#     folds and scores them on the fifth, rotating, so the number reported is what this
#     procedure produces on rows it has not seen.
from itertools import product

from scipy.optimize import minimize_scalar
from sklearn.metrics import log_loss

# The linear model again, on the same folds, but keeping decision_function instead of the
# predicted label: a label cannot be averaged with another model's logits, a margin can.
BEST_VIEWS, BEST_C = ["word", "char", "emb"], 0.05
linear_oof = np.zeros((len(train), len(LABELS)), dtype=np.float32)
for fold in range(N_SPLITS):
    is_validation = fold_ids == fold
    X_fit, X_apply = build_features(texts[~is_validation], texts[is_validation],
                                    train_embeddings[~is_validation],
                                    train_embeddings[is_validation], BEST_VIEWS)
    fitted = LinearSVC(C=BEST_C, class_weight="balanced",
                       random_state=RANDOM_STATE, max_iter=10000).fit(X_fit, y_train[~is_validation])
    linear_oof[is_validation] = fitted.decision_function(X_apply)
X_fit, X_apply = build_features(texts, test["text"].values,
                                train_embeddings, test_embeddings, BEST_VIEWS)
linear_test = LinearSVC(C=BEST_C, class_weight="balanced", random_state=RANDOM_STATE,
                        max_iter=10000).fit(X_fit, y_train).decision_function(X_apply)
print(f"linear model, same folds: macro F1 "
      f"{f1_score(y_index, linear_oof.argmax(1), average='macro'):.4f}")

COMPONENTS = {"base_avg": (base_avg_oof, base_avg_test),
              "large":    (large_oof,    large_test),
              "linear":   (linear_oof,   linear_test)}


def softmax(scores, temperature=1.0):
    shifted = scores / temperature
    shifted = shifted - shifted.max(axis=1, keepdims=True)
    exponentiated = np.exp(shifted)
    return exponentiated / exponentiated.sum(axis=1, keepdims=True)


temperatures = {}
print("\ntemperature fitted on out-of-fold log loss")
for name, (oof, _) in COMPONENTS.items():
    fitted = minimize_scalar(
        lambda t: log_loss(y_index, softmax(oof, t), labels=list(range(len(LABELS)))),
        bounds=(0.05, 20.0), method="bounded")
    temperatures[name] = fitted.x
    print(f"  {name:9s} T={fitted.x:5.2f}   log loss {fitted.fun:.4f}"
          f"   own macro F1 {f1_score(y_index, oof.argmax(1), average='macro'):.4f}")

names = list(COMPONENTS)
WEIGHT_GRID = [np.array(list(w) + [1.0 - sum(w)])
               for w in product(np.arange(0, 1.01, 0.05), repeat=len(names) - 1)
               if sum(w) <= 1.0 + 1e-9]


def mix(rows, weights, which):
    """Weighted average of the calibrated probabilities, for the given rows."""
    return sum(weights[i] * softmax(COMPONENTS[n][which][rows], temperatures[n])
               for i, n in enumerate(names))


def best_weights(rows):
    scored = [(f1_score(y_index[rows], mix(rows, w, 0).argmax(1), average="macro"), w)
              for w in WEIGHT_GRID]
    return max(scored, key=lambda pair: pair[0])


everything = np.ones(len(y_index), dtype=bool)
optimistic, chosen_weights = best_weights(everything)
print("\nweights chosen on everything:",
      {n: round(float(chosen_weights[i]), 2) for i, n in enumerate(names)},
      f"-> macro F1 {optimistic:.4f} (optimistic)")

held_out = np.empty(len(y_index), dtype=int)
for fold in range(N_SPLITS):
    _, weights = best_weights(fold_ids != fold)
    held_out[fold_ids == fold] = mix(fold_ids == fold, weights, 0).argmax(1)
honest = f1_score(y_index, held_out, average="macro")
print(f"nested (weights from four folds, scored on the fifth): macro F1 {honest:.4f}")
for name, (oof, _) in COMPONENTS.items():
    own = f1_score(y_index, oof.argmax(1), average="macro")
    print(f"  vs {name:9s} {own:.4f}   gain {honest - own:+.4f}"
          f"   ({(honest - own) * len(test):+.0f} of {len(test)} test rows)")

blend_test = sum(chosen_weights[i] * softmax(COMPONENTS[n][1], temperatures[n])
                 for i, n in enumerate(names))
blend_labels = [LABELS[i] for i in blend_test.argmax(1)]
submission = check_and_write(test["post_id"].values, blend_labels, "blend_calibrated")
print(f"\nFINAL ENTRY: nested out-of-fold macro F1 {honest:.4f}")
print(submission["label"].value_counts().to_string())


linear model, same folds: macro F1 0.5708

temperature fitted on out-of-fold log loss
  base_avg  T= 1.50   log loss 1.0462   own macro F1 0.6120
  large     T= 1.87   log loss 1.0552   own macro F1 0.6101
  linear    T= 0.28   log loss 1.1614   own macro F1 0.5708

weights chosen on everything: {'base_avg': 0.25, 'large': 0.25, 'linear': 0.5} -> macro F1 0.6288 (optimistic)
nested (weights from four folds, scored on the fifth): macro F1 0.6236
  vs base_avg  0.6120   gain +0.0117   (+17 of 1440 test rows)
  vs large     0.6101   gain +0.0135   (+19 of 1440 test rows)
  vs linear    0.5708   gain +0.0529   (+76 of 1440 test rows)
all five checks passed for 'blend_calibrated' (1440 rows, 6 distinct labels)
   written: ..\_workbench\kaggle\submissions\blend_calibrated__id.csv
   written: ..\_workbench\kaggle\submissions\blend_calibrated__post_id.csv

FINAL ENTRY: nested out-of-fold macro F1 0.6236
label
joy         406
anger       324
sadness     255
surprise    181
disgust     148
fear 

### 3.6 Starting from models that already know emotions

**Used in one of the two final entries (through 3.7).** These checkpoints were fine-tuned by their authors on GoEmotions, and on 17 November I found that 48.8% of the test texts are GoEmotions comments whose annotator labels agree with the competition labels 95.3% of the time (measured on the training set), so these models have in effect seen part of the test answers. I asked the TAs, and they replied that it is fine as long as I train my own model with the data provided by the competition. So I use them in one final entry and keep a second final entry without them (3.8); the report gives both scores.

Everything in 3.4 started from a model that had read 154 million tweets but had never been told what an emotion is: its classifier head began as random numbers. The checkpoints in this section were already fine-tuned on emotion datasets, and their heads already separate anger, disgust, fear, joy, sadness and surprise, alongside classes this task does not have, such as neutral. Keeping the head rows for our six labels means training starts from a model that can already tell the emotions apart, and our 3,360 tweets only have to adapt it to this corpus.

**GPU kernel again.** The five checkpoints below take about 45 minutes in total on an RTX 5090, most of it the two large models.

In [14]:
# Loading an emotion checkpoint without throwing its head away.
#
# finetune_cv in 3.4 asks for num_labels=6 with ignore_mismatched_sizes=True, which
# discards any head whose size differs from six and starts a new one at random. That
# is right for a checkpoint with no head worth keeping, and wrong here: a 7-class
# emotion head already holds a direction for each of our six emotions. So the head is
# cut down to our six rows instead of replaced. The training loop is otherwise the one
# from 3.4, line for line, so the numbers stay comparable with that section.

# 3.6 and 3.7 reuse what 3.4 and 3.5 computed in this kernel session. If the kernel was
# restarted since then, stop here and run the notebook from the top first.
for needed in ("finetune_cv", "SEEDS", "base_avg_oof", "linear_oof", "honest", "check_and_write"):
    if needed not in globals():
        raise NameError(f"{needed} is not defined: run sections 1 to 3.5 above first")

def load_with_emotion_head(model_name):
    """Load a checkpoint and keep only the classifier rows for our six labels."""
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    # id2label maps each output row of the original head to its label name.
    source_row = {str(name).lower(): int(row) for row, name in model.config.id2label.items()}
    missing = [label for label in LABELS if label not in source_row]
    if missing:
        raise ValueError(f"{model_name} has no output for {missing}")
    rows = [source_row[label] for label in LABELS]          # reordered to match LABELS
    old = model.classifier.out_proj
    new = torch.nn.Linear(old.in_features, len(rows))
    new.weight.data = old.weight.data[rows].clone()         # one weight row per kept label
    new.bias.data = old.bias.data[rows].clone()
    model.classifier.out_proj = new
    model.config.num_labels = model.num_labels = len(rows)
    return model.cuda()


def finetune_emotion_cv(model_name, seed=RANDOM_STATE, epochs=5, lr=3e-5, batch_size=32, max_len=96):
    """finetune_cv from 3.4 with one change: the model keeps its emotion head."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    oof = np.zeros((len(train), len(LABELS)), dtype=np.float32)
    test_logits = np.zeros((len(test), len(LABELS)), dtype=np.float32)
    started = time.time()

    for fold in range(N_SPLITS):
        is_validation = fold_ids == fold
        tokeniser = AutoTokenizer.from_pretrained(model_name)
        model = load_with_emotion_head(model_name)          # the only line that differs

        loader = DataLoader(TweetDataset(texts[~is_validation], y_index[~is_validation],
                                         tokeniser, max_len),
                            batch_size=batch_size, shuffle=True)
        optimiser = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
        schedule = torch.optim.lr_scheduler.OneCycleLR(
            optimiser, max_lr=lr, total_steps=len(loader) * epochs, pct_start=0.1)

        model.train()
        for _ in range(epochs):
            for batch in loader:
                batch = {k: v.cuda() for k, v in batch.items()}
                with torch.autocast("cuda", dtype=torch.bfloat16):
                    logits = model(**{k: v for k, v in batch.items() if k != "labels"}).logits
                    loss = F.cross_entropy(logits, batch["labels"])
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimiser.step()
                schedule.step()
                optimiser.zero_grad(set_to_none=True)

        oof[is_validation] = predict_logits(
            model, DataLoader(TweetDataset(texts[is_validation], None, tokeniser, max_len),
                              batch_size=128))
        test_logits += predict_logits(
            model, DataLoader(TweetDataset(test["text"].values, None, tokeniser, max_len),
                              batch_size=128)) / N_SPLITS
        print(f"  fold {fold}: macro F1 "
              f"{f1_score(y_index[is_validation], oof[is_validation].argmax(1), average='macro'):.4f}",
              flush=True)
        del model
        torch.cuda.empty_cache()

    score = f1_score(y_index, oof.argmax(1), average="macro")
    print(f"  out-of-fold macro F1 {score:.4f}  ({time.time() - started:.0f}s)", flush=True)
    return oof, test_logits, score


In [15]:
# Two checkpoints from the same author, three seeds each, averaged the way the base
# model was in 3.4. Both were fine-tuned on six English emotion datasets with seven
# classes, our six plus neutral; one is distilled (82 million parameters), the other
# large (355 million). The large one takes the smaller learning rate and batch that
# the large model in 3.4 needed.
EMOTION_RUNS = {
    "emo_distil": ("j-hartmann/emotion-english-distilroberta-base", dict(lr=3e-5, batch_size=32)),
    "emo_large":  ("j-hartmann/emotion-english-roberta-large",      dict(lr=1.5e-5, batch_size=16)),
}

emotion_oof, emotion_test = {}, {}
for tag, (checkpoint, settings) in EMOTION_RUNS.items():
    runs = []
    for seed in SEEDS:
        print(f"{checkpoint}  seed={seed}")
        runs.append(finetune_emotion_cv(checkpoint, seed=seed, **settings))
    # Keep the first seed on its own as well: the four-model blend in 3.7 uses it.
    emotion_oof[tag + "_s42"], emotion_test[tag + "_s42"] = runs[0][0], runs[0][1]
    emotion_oof[tag + "_avg"] = np.mean([r[0] for r in runs], axis=0)
    emotion_test[tag + "_avg"] = np.mean([r[1] for r in runs], axis=0)
    seed_scores = [r[2] for r in runs]
    averaged = f1_score(y_index, emotion_oof[tag + "_avg"].argmax(1), average="macro")
    print(f"\n{tag}: seeds {[round(s, 4) for s in seed_scores]}"
          f" (spread {max(seed_scores) - min(seed_scores):.4f}), averaged {averaged:.4f}\n")

print(f"tweet-domain base, seed-averaged (3.4): {base_avg_score:.4f}")
for tag in EMOTION_RUNS:
    averaged = f1_score(y_index, emotion_oof[tag + "_avg"].argmax(1), average="macro")
    print(f"{tag:10s} seed-averaged         : {averaged:.4f}   ({averaged - base_avg_score:+.4f})")


j-hartmann/emotion-english-distilroberta-base  seed=42


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 26258.16it/s]


  fold 0: macro F1 0.6252


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 26250.34it/s]


  fold 1: macro F1 0.6179


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 35033.17it/s]


... (51 outputs left out)

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 38625.93it/s]


  fold 3: macro F1 0.6477


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 26023.20it/s]


  fold 4: macro F1 0.5862
  out-of-fold macro F1 0.6214  (492s)

emo_large: seeds [0.6173, 0.6253, 0.6214] (spread 0.0080), averaged 0.6242

tweet-domain base, seed-averaged (3.4): 0.6120
emo_distil seed-averaged         : 0.6262   (+0.0143)
emo_large  seed-averaged         : 0.6242   (+0.0122)


In [16]:
# Three more emotion checkpoints, one seed each. They were trained on different text,
# which is the point: a blend gains only where its members are wrong about different rows.
#   go_emotions   - Reddit comments, 28 emotion classes
#   michellejieli - dialogue from the TV series Friends, 7 classes
#   cardiffnlp    - tweets again, 11 emotion classes, large model
MORE_EMOTION = {
    "emo_goemo":     ("SamLowe/roberta-base-go_emotions",                 dict(lr=3e-5, batch_size=32)),
    "emo_michelle":  ("michellejieli/emotion_text_classifier",            dict(lr=3e-5, batch_size=32)),
    "emo_cardlarge": ("cardiffnlp/twitter-roberta-large-emotion-latest",  dict(lr=1.5e-5, batch_size=16)),
}
for tag, (checkpoint, settings) in MORE_EMOTION.items():
    print(checkpoint)
    emotion_oof[tag], emotion_test[tag], score = finetune_emotion_cv(checkpoint, **settings)
    print(f"{tag}: {score:.4f}\n")


SamLowe/roberta-base-go_emotions


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 880.20it/s]


  fold 0: macro F1 0.6416


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1112.89it/s]


  fold 1: macro F1 0.6203


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1201.67it/s]


... (21 outputs left out)

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 509.73it/s]


  fold 3: macro F1 0.6192


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 647.43it/s]


  fold 4: macro F1 0.5807
  out-of-fold macro F1 0.6059  (676s)
emo_cardlarge: 0.6059



### 3.7 Blends with the emotion checkpoints (the seven-model file is a final entry)

**The seven-model file written here is one of my two final entries.** The other final entry, without the emotion checkpoints, comes from 3.5 and 3.8. The four-model file is not a final entry.

The same calibrated blend as 3.5, rebuilt with the emotion models. Two versions are written. The four-model one mirrors the entry that had the best public score at the time; the seven-model one has the best nested out-of-fold score.

In [17]:
# The blend from 3.5, wrapped in a function so it can be run on any set of models.
# Same three steps: one temperature per model fitted on out-of-fold log loss, a weight
# grid searched on macro F1, and a nested estimate (weights from four folds, scored on
# the fifth). The grid step is 0.05 for four models and 0.1 for seven, because the
# number of weight combinations grows quickly with the number of models.

def calibrated_blend(components, step):
    names = list(components)
    temps = {}
    for n in names:
        oof_scores = components[n][0]
        temps[n] = minimize_scalar(
            lambda t: log_loss(y_index, softmax(oof_scores, t), labels=list(range(len(LABELS)))),
            bounds=(0.05, 20.0), method="bounded").x
    p_oof = [softmax(components[n][0], temps[n]) for n in names]
    p_test = [softmax(components[n][1], temps[n]) for n in names]
    grid = [np.array(list(w) + [1.0 - sum(w)])
            for w in product(np.arange(0, 1.0 + 1e-9, step), repeat=len(names) - 1)
            if sum(w) <= 1.0 + 1e-9]

    def pick(rows):
        scored = [(f1_score(y_index[rows], sum(w[i] * p_oof[i][rows] for i in range(len(names))).argmax(1),
                            average="macro"), w) for w in grid]
        return max(scored, key=lambda pair: pair[0])

    optimistic, weights = pick(np.ones(len(y_index), dtype=bool))
    held = np.empty(len(y_index), dtype=int)
    per_fold = []
    for fold in range(N_SPLITS):
        _, w = pick(fold_ids != fold)
        rows = fold_ids == fold
        held[rows] = sum(w[i] * p_oof[i][rows] for i in range(len(names))).argmax(1)
        per_fold.append(f1_score(y_index[rows], held[rows], average="macro"))
    nested = f1_score(y_index, held, average="macro")

    print(f"{len(names)} models, grid step {step}")
    for i, n in enumerate(names):
        own = f1_score(y_index, components[n][0].argmax(1), average="macro")
        print(f"  {n:14s} own {own:.4f}   T={temps[n]:5.2f}   weight {weights[i]:.2f}")
    print(f"  optimistic {optimistic:.4f} | nested {nested:.4f} | per fold {np.round(per_fold, 4)}")
    test_probs = sum(weights[i] * p_test[i] for i in range(len(names)))
    return nested, per_fold, test_probs


FOUR = {"base_avg":   (base_avg_oof, base_avg_test),
        "emo_large":  (emotion_oof["emo_large_s42"], emotion_test["emo_large_s42"]),
        "emo_distil": (emotion_oof["emo_distil_s42"], emotion_test["emo_distil_s42"]),
        "linear":     (linear_oof, linear_test)}
SEVEN = {"base_avg":       (base_avg_oof, base_avg_test),
         "emo_large_avg":  (emotion_oof["emo_large_avg"], emotion_test["emo_large_avg"]),
         "emo_distil_avg": (emotion_oof["emo_distil_avg"], emotion_test["emo_distil_avg"]),
         "linear":         (linear_oof, linear_test),
         "emo_goemo":      (emotion_oof["emo_goemo"], emotion_test["emo_goemo"]),
         "emo_michelle":   (emotion_oof["emo_michelle"], emotion_test["emo_michelle"]),
         "emo_cardlarge":  (emotion_oof["emo_cardlarge"], emotion_test["emo_cardlarge"])}

print(f"3.5 blend without emotion models, nested: {honest:.4f}\n")
four_nested, four_folds, four_probs = calibrated_blend(FOUR, 0.05)
print()
seven_nested, seven_folds, seven_probs = calibrated_blend(SEVEN, 0.1)

print(f"\nfour models  vs 3.5: {four_nested - honest:+.4f}  = {(four_nested - honest) * len(test):+.0f} of {len(test)} test rows")
print(f"seven models vs four: {seven_nested - four_nested:+.4f}  = {(seven_nested - four_nested) * len(test):+.0f} rows,"
      f" up in {sum(s > f for s, f in zip(seven_folds, four_folds))} of {N_SPLITS} folds")

print()
for stem, probs in (("final_emo4", four_probs), ("final_emo7", seven_probs)):
    labels_out = [LABELS[i] for i in probs.argmax(1)]
    written = check_and_write(test["post_id"].values, labels_out, stem)
    print("   ", written["label"].value_counts().to_dict())


3.5 blend without emotion models, nested: 0.6236

4 models, grid step 0.05
  base_avg       own 0.6120   T= 1.50   weight 0.25
  emo_large      own 0.6173   T= 2.03   weight 0.15
  emo_distil     own 0.6169   T= 1.72   weight 0.35
  linear         own 0.5708   T= 0.28   weight 0.25
  optimistic 0.6470 | nested 0.6438 | per fold [0.6492 0.6337 0.6562 0.6579 0.62  ]

7 models, grid step 0.1
  base_avg       own 0.6120   T= 1.50   weight 0.10
  emo_large_avg  own 0.6242   T= 1.98   weight 0.20
  emo_distil_avg own 0.6262   T= 1.70   weight 0.20
  linear         own 0.5708   T= 0.28   weight 0.10
  emo_goemo      own 0.6222   T= 1.78   weight 0.20
  emo_michelle   own 0.6014   T= 1.73   weight 0.00
  emo_cardlarge  own 0.6059   T= 2.49   weight 0.20
  optimistic 0.6555 | nested 0.6536 | per fold [0.6713 0.6403 0.652  0.672  0.63  ]

four models  vs 3.5: +0.0201  = +29 of 1440 test rows
seven models vs four: +0.0098  = +14 rows, up in 4 of 5 folds

all five checks passed for 'final_emo4' (1

### 3.8 The final entry without the emotion checkpoints

My second final entry uses only models that learned from the competition data and from unlabelled text: the 3.5 blend of the tweet-domain base model (three seeds), the tweet-domain large model and the linear model. The emotion checkpoints of 3.6 are used only in the other final entry (3.7), for the reason given at the top of 3.6.

One more step is applied to the 3.5 blend. Fifty-nine test tweets appear word for word in the training set, each time with a single label there, so the cell below gives those rows the training label. Training texts that carry two different labels are left alone, because there the training set itself does not know the answer.

The cell reads the file written by 3.5 and needs no GPU and no variables from earlier cells. It is meant to be run on its own in a fresh kernel, so its counter starts again at 1.

In [1]:
# Final clean entry: test rows whose text appears word for word in the training set take the
# training label, but only when every copy in the training set carries the same label.
#
# Why this is allowed: the training labels are the data we were given. Finding that a test
# tweet is literally a training tweet is data analysis, not outside information.
# Why only consistent groups: 16 training texts appear with two different labels; for those
# the training set itself does not know the answer, so the model's prediction is kept.
# This cell needs no GPU and no variables from earlier cells, so it can be run on its own.
import json
import re
from pathlib import Path

import pandas as pd

DATA_DIR = Path("data/kaggle")
SUBMISSION_DIR = Path("../_workbench/kaggle/submissions")

def normalise(text):                      # same normalisation as the fold split in T0 step 4
    return re.sub(r"\s+", " ", str(text).lower()).strip()

posts = pd.DataFrame(json.loads((DATA_DIR / "final_posts.json").read_text(encoding="utf-8")))
ident = pd.read_csv(DATA_DIR / "data_identification.csv")
emotion = pd.read_csv(DATA_DIR / "emotion.csv")
sample = pd.read_csv(DATA_DIR / "sampleSubmission.csv")
df = posts.merge(ident, on="post_id", how="inner")
train = df[df["set"] == "train"].merge(emotion, on="post_id", how="inner")
test = df[df["set"] == "test"]

groups = train.assign(key=train["text"].map(normalise)).groupby("key")["label"].unique()
consistent = {k: v[0] for k, v in groups.items() if len(v) == 1}
conflicting = {k for k, v in groups.items() if len(v) > 1}
test_key = test.set_index("post_id")["text"].map(normalise)
print(f"training texts that occur more than once: {int((train['text'].map(normalise).value_counts() > 1).sum())}, "
      f"of which {len(conflicting)} carry two different labels")
print(f"test rows identical to a training text: {int(test_key.isin(consistent).sum() + test_key.isin(conflicting).sum())} "
      f"({int(test_key.isin(consistent).sum())} with a consistent label, {int(test_key.isin(conflicting).sum())} in a conflicting group)")

blend = pd.read_csv(SUBMISSION_DIR / "blend_calibrated__id.csv")      # written by 3.5
train_label = blend["id"].map(test_key).map(consistent)
changed = train_label.notna() & (train_label != blend["label"])
print(f"the 3.5 blend already agrees with {int((train_label.notna() & ~changed).sum())} of them; rows changed: {int(changed.sum())}")
for pid, old, new in zip(blend.loc[changed, "id"], blend.loc[changed, "label"], train_label[changed]):
    print(f"   {pid}: {old} -> {new}")
final = blend.copy()
final.loc[changed, "label"] = train_label[changed]

# the five checks used for every entry in this notebook
assert len(final) == len(sample) == 1440, "row count"
assert (final["id"].to_numpy() == sample["post_id"].to_numpy()).all(), "id order"
assert final["label"].notna().all(), "missing labels"
assert set(final["label"]) <= set(train["label"].unique()), "unknown label"
assert final["label"].nunique() == 6, "a class disappeared"
for header in ("id", "post_id"):
    path = SUBMISSION_DIR / f"final_clean__{header}.csv"
    final.rename(columns={"id": header}).to_csv(path, index=False)
    print("   written:", path)
print("FINAL CLEAN ENTRY:", final["label"].value_counts().to_dict())

training texts that occur more than once: 50, of which 16 carry two different labels
test rows identical to a training text: 59 (59 with a consistent label, 0 in a conflicting group)
the 3.5 blend already agrees with 56 of them; rows changed: 3
   p8otxou: surprise -> anger
   4i2nr1p: fear -> disgust
   68q0j9m: anger -> sadness
   written: ..\_workbench\kaggle\submissions\final_clean__id.csv
   written: ..\_workbench\kaggle\submissions\final_clean__post_id.csv
FINAL CLEAN ENTRY: {'joy': 406, 'anger': 324, 'sadness': 256, 'surprise': 180, 'disgust': 149, 'fear': 125}
